# 📊 Phase 2 — T8: Fading Etkilerinin IQ Sinyal Görselleştirmesi

Bu notebook, orijinal AWGN ve fading uygulanmış dataset'leri yükleyerek kapsamlı görsel analiz sunar.

**Görselleştirmeler:**
1. Constellation diyagramları (tüm modülasyonlar × tüm kanallar)
2. Zaman serisi karşılaştırması (I/Q dalga formları)
3. SNR vs Kanal etkisi grid matrisi
4. Güç dağılımı analizi (histogram)
5. Kanal bozulma ısı haritası (heatmap)

**Ön koşul:** `04_generate_faded_datasets.ipynb` çalıştırılmış ve Drive'da faded pkl dosyaları mevcut olmalı.

## 1. Ortam Kurulumu

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import os

# Görsel kalite ayarları
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.facecolor': 'white'
})

print('✅ Kütüphaneler yüklendi.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/erigami-sl/AMR-UnderDifferentNoises-DL.git'
PROJECT_DIR = '/content/AMR-UnderDifferentNoises-DL'

if not os.path.exists(PROJECT_DIR):
    !git clone -b dev {REPO_URL}
else:
    !cd {PROJECT_DIR} && git pull origin dev

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f'✅ Çalışma dizini: {os.getcwd()}')

## 2. Dataset'leri Yükleme

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/AMR-Project'

FILES = {
    'AWGN (Orijinal)': 'RML2016.10a_dict.pkl',
    'Rayleigh':        'RML2016.10a_rayleigh.pkl',
    'Rician K=3':      'RML2016.10a_rician_K3.pkl',
    'Rician K=10':     'RML2016.10a_rician_K10.pkl'
}

datasets = {}
for label, fname in FILES.items():
    path = os.path.join(DRIVE_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f'{fname} bulunamadı: {path}')
    datasets[label] = pickle.load(open(path, 'rb'), encoding='iso-8859-1')
    print(f'  ✅ {label:20s} — {len(datasets[label])} anahtar')

mods = sorted(list(set([k[0] for k in datasets['AWGN (Orijinal)'].keys()])))
snrs = sorted(list(set([k[1] for k in datasets['AWGN (Orijinal)'].keys()])))
channel_names = list(datasets.keys())
channel_colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50']

print(f'\n📡 Modülasyonlar ({len(mods)}): {mods}')
print(f'📶 SNR: {snrs[0]} → {snrs[-1]} dB ({len(snrs)} seviye)')

## 3. Constellation Diyagramları — Tüm Modülasyonlar × Tüm Kanallar

Yüksek SNR'de (10 dB) her modülasyonun constellation diyagramı, kanal etkisini net gösterir.

In [ ]:
SNR_VIS = 10
N_SHOW = 40  # Üst üste bindirilecek örnek sayısı

fig, axes = plt.subplots(len(mods), len(channel_names), figsize=(20, 3 * len(mods)))

for row, mod in enumerate(mods):
    for col, (ch_name, ch_data) in enumerate(datasets.items()):
        ax = axes[row, col]
        X = ch_data[(mod, SNR_VIS)]
        I = X[:N_SHOW, 0, :].flatten()
        Q = X[:N_SHOW, 1, :].flatten()
        ax.scatter(I, Q, s=0.5, alpha=0.12, color=channel_colors[col])
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)
        ax.tick_params(labelsize=7)
        if row == 0:
            ax.set_title(ch_name, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(mod, fontsize=10, fontweight='bold')

fig.suptitle(f'Constellation Diyagramları — SNR={SNR_VIS} dB (11 Modülasyon × 4 Kanal)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Zaman Serisi Karşılaştırması

Tek bir örneğin I/Q dalga formunun kanal bazında nasıl bozulduğunu gösterir.

In [ ]:
VIS_MODS = ['QPSK', 'QAM16', 'QAM64', '8PSK']
SAMPLE_IDX = 0

for mod_name in VIS_MODS:
    fig, axes = plt.subplots(2, len(channel_names), figsize=(20, 5))
    key = (mod_name, SNR_VIS)

    for col, (ch_name, ch_data) in enumerate(datasets.items()):
        X = ch_data[key]
        # I kanalı
        axes[0, col].plot(X[SAMPLE_IDX, 0, :], color=channel_colors[col], linewidth=0.8, alpha=0.9)
        axes[0, col].set_title(ch_name, fontsize=10, fontweight='bold')
        axes[0, col].grid(True, alpha=0.2)
        if col == 0:
            axes[0, col].set_ylabel('I Kanalı', fontweight='bold')
        # Q kanalı
        axes[1, col].plot(X[SAMPLE_IDX, 1, :], color=channel_colors[col], linewidth=0.8, alpha=0.9)
        axes[1, col].grid(True, alpha=0.2)
        axes[1, col].set_xlabel('Örnek İndeksi')
        if col == 0:
            axes[1, col].set_ylabel('Q Kanalı', fontweight='bold')

    fig.suptitle(f'{mod_name} — SNR={SNR_VIS} dB — I/Q Zaman Serisi', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    print()

## 5. SNR vs Kanal Etkisi Grid Matrisi

QPSK için farklı SNR seviyeleri ve kanal koşullarında constellation dağılımı.

In [ ]:
GRID_MOD = 'QPSK'
GRID_SNRS = [-20, -10, -4, 0, 6, 10, 18]

fig, axes = plt.subplots(len(GRID_SNRS), len(channel_names), figsize=(18, 3.5 * len(GRID_SNRS)))

for row, snr_val in enumerate(GRID_SNRS):
    for col, (ch_name, ch_data) in enumerate(datasets.items()):
        ax = axes[row, col]
        X = ch_data[(GRID_MOD, snr_val)]
        I = X[:30, 0, :].flatten()
        Q = X[:30, 1, :].flatten()
        ax.scatter(I, Q, s=0.5, alpha=0.12, color=channel_colors[col])
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)
        ax.tick_params(labelsize=7)
        if row == 0:
            ax.set_title(ch_name, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'SNR={snr_val} dB', fontsize=10, fontweight='bold')

fig.suptitle(f'{GRID_MOD} — SNR vs Kanal Etkisi Matrisi', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Sinyal Gücü Dağılımı — Histogram Analizi

Fading'in sinyal gücü üzerindeki etkisini histogram ile gösterir.  
Rayleigh'de geniş bir güç dağılımı (derin sönümleme), Rician'da daha dar dağılım beklenir.

In [ ]:
HIST_MOD = 'QPSK'
HIST_SNR = 10

fig, axes = plt.subplots(1, len(channel_names), figsize=(20, 4), sharey=True)

for col, (ch_name, ch_data) in enumerate(datasets.items()):
    X = ch_data[(HIST_MOD, HIST_SNR)]
    # Örnek başına ortalama güç
    power_per_sample = np.mean(X[:, 0, :]**2 + X[:, 1, :]**2, axis=1)

    axes[col].hist(power_per_sample, bins=50, color=channel_colors[col], alpha=0.75, edgecolor='white', linewidth=0.5)
    axes[col].set_title(f'{ch_name}\nμ={np.mean(power_per_sample):.4f}, σ={np.std(power_per_sample):.4f}', fontsize=10)
    axes[col].set_xlabel('Ortalama Güç (I²+Q²)')
    axes[col].axvline(np.mean(power_per_sample), color='black', linestyle='--', linewidth=1, alpha=0.6)
    axes[col].grid(True, alpha=0.2)

axes[0].set_ylabel('Frekans')
fig.suptitle(f'{HIST_MOD} SNR={HIST_SNR} dB — Örnek Başına Güç Dağılımı', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Kanal Bozulma Isı Haritası (Heatmap)

Her (modülasyon, SNR) çifti için fading sonrası güç oranını (`P_faded / P_awgn`) gösterir.  
Rayleigh'in hangi modülasyonlarda daha fazla bozulma yaptığını analiz eder.

In [ ]:
fading_channels = {k: v for k, v in datasets.items() if k != 'AWGN (Orijinal)'}

fig, axes = plt.subplots(1, len(fading_channels), figsize=(20, 6))

for idx, (ch_name, ch_data) in enumerate(fading_channels.items()):
    # Güç oranı matrisi: (n_mods x n_snrs)
    ratio_matrix = np.zeros((len(mods), len(snrs)))
    for i, mod in enumerate(mods):
        for j, snr in enumerate(snrs):
            key = (mod, snr)
            p_awgn  = np.mean(datasets['AWGN (Orijinal)'][key]**2)
            p_faded = np.mean(ch_data[key]**2)
            ratio_matrix[i, j] = p_faded / p_awgn if p_awgn > 0 else 1.0

    im = axes[idx].imshow(ratio_matrix, aspect='auto', cmap='RdYlGn',
                          vmin=0.3, vmax=1.7, interpolation='nearest')
    axes[idx].set_title(ch_name, fontsize=12, fontweight='bold')
    axes[idx].set_xticks(range(0, len(snrs), 2))
    axes[idx].set_xticklabels([str(s) for s in snrs[::2]], fontsize=8)
    axes[idx].set_xlabel('SNR (dB)')
    axes[idx].set_yticks(range(len(mods)))
    axes[idx].set_yticklabels(mods, fontsize=8)
    if idx == 0:
        axes[idx].set_ylabel('Modülasyon')
    plt.colorbar(im, ax=axes[idx], shrink=0.8, label='P_faded / P_awgn')

fig.suptitle('Kanal Bozulma Isı Haritası — Güç Oranı (Faded / AWGN)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('💡 Yeşil ≈ 1.0 (güç korunuyor), Kırmızı < 1.0 (güç kaybı), Sarı > 1.0 (güç artışı)')

## 8. Özet ve Bulgular

In [ ]:
print('=' * 60)
print('📊 T8 — FADING ETKİSİ GÖRSELLEŞTİRME ÖZETİ')
print('=' * 60)
print()
print('✅ Üretilen görselleştirmeler:')
print('  1. Constellation (11 mod × 4 kanal) grid')
print('  2. I/Q zaman serisi karşılaştırması')
print('  3. SNR vs kanal etkisi matrisi')
print('  4. Güç dağılımı histogramları')
print('  5. Bozulma ısı haritası (heatmap)')
print()
print('📌 Önemli gözlemler:')
print('  • Rayleigh fading constellation noktalarını en fazla dağıtır')
print('  • Rician K=10 orijinale en yakın davranışı gösterir')
print('  • Düşük SNR + Rayleigh kombinasyonu sınıflandırma için en zor senaryo')
print('  • QAM16/QAM64 gibi yoğun constellation modülasyonları fading\'e en hassas')
print()
print('✅ Sonraki adım: T9 — Bu dataset\'ler ile model eğitimi')

---
## ✅ T8 Tamamlandı

Fading etkilerinin IQ sinyalleri üzerindeki görsel analizi tamamlandı.

**Sonraki adım:** T9 — Fading dataset'leri üzerinde MCLDNN ve PET-CGDNN eğitimleri.